In [1]:
print("Let's Start")

Let's Start


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [3]:
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv')

In [4]:
X = df.drop(columns=['price'])
y = df['price']

In [5]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

In [6]:
# import automac
# afe  = automac.feature_engineering.FeatureEngineering(X , y_transformed)


In [7]:
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline


In [8]:
from category_encoders import TargetEncoder
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [53]:
X.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,3.0,2.0,2,New Property,850.0,0.0,0.0,0.0,Low,Low Floor
1,flat,sector 89,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Mid Floor
2,flat,sohna road,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low,High Floor
3,flat,sector 92,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High,Mid Floor
4,flat,sector 102,2.0,2.0,1,Relatively New,582.0,0.0,1.0,0.0,High,Mid Floor


In [54]:
x_transed , encoder = afe.encode_categories(['agePossession' , 'luxury_category' , 'floor_category' , 'sector' , 'property_type' , 'balcony'])

In [55]:
x_s , scaler = afe.scale_features(['built_up_area'])

In [59]:
x_s.info()

<class 'pandas.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3554 non-null   float64
 1   sector           3554 non-null   float64
 2   bedRoom          3554 non-null   float64
 3   bathroom         3554 non-null   float64
 4   balcony          3554 non-null   float64
 5   agePossession    3554 non-null   float64
 6   built_up_area    3554 non-null   float64
 7   servant room     3554 non-null   float64
 8   store room       3554 non-null   float64
 9   furnishing_type  3554 non-null   float64
 10  luxury_category  3554 non-null   float64
 11  floor_category   3554 non-null   float64
dtypes: float64(12)
memory usage: 333.3 KB


In [60]:
from sklearn.linear_model import LinearRegression , Ridge , Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor , RandomForestRegressor , AdaBoostRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [79]:
models = [LinearRegression() , Ridge() , Lasso() , DecisionTreeRegressor() , SVR() , GradientBoostingRegressor() , RandomForestRegressor() , AdaBoostRegressor() , XGBRegressor() , CatBoostRegressor() , LGBMRegressor()]
names   = ['LinearRegression' , 'Ridge' , 'Lasso' , 'DecisionTreeRegressor' , 'SVR' , 'GradientBoostingRegressor' , 'RandomForestRegressor' , 'AdaBoostRegressor' , 'XGBRegressor' , 'CatBoostRegressor' , 'LGBMRegressor']

In [80]:
ra = automac.analyser.RegressionAnalyser(x=x_s , y=y_transformed)


In [82]:
scores = {}
for i in range(len(models)):
    score = ra.regression_metrics(model=models[i])
    scores[names[i]] = score

Learning rate set to 0.047282
0:	learn: 0.5373051	total: 1.65ms	remaining: 1.65s
1:	learn: 0.5197666	total: 2.48ms	remaining: 1.24s
2:	learn: 0.5022029	total: 3.29ms	remaining: 1.09s
3:	learn: 0.4848223	total: 4.98ms	remaining: 1.24s
4:	learn: 0.4688345	total: 5.75ms	remaining: 1.15s
5:	learn: 0.4534298	total: 6.49ms	remaining: 1.07s
6:	learn: 0.4390343	total: 7.13ms	remaining: 1.01s
7:	learn: 0.4250292	total: 7.92ms	remaining: 982ms
8:	learn: 0.4110929	total: 8.69ms	remaining: 957ms
9:	learn: 0.3990776	total: 9.5ms	remaining: 941ms
10:	learn: 0.3872630	total: 10.4ms	remaining: 934ms
11:	learn: 0.3758566	total: 11.2ms	remaining: 921ms
12:	learn: 0.3655315	total: 11.9ms	remaining: 903ms
13:	learn: 0.3555138	total: 12.8ms	remaining: 903ms
14:	learn: 0.3461240	total: 13.6ms	remaining: 895ms
15:	learn: 0.3368887	total: 14.5ms	remaining: 890ms
16:	learn: 0.3284814	total: 15.3ms	remaining: 885ms
17:	learn: 0.3206818	total: 16.2ms	remaining: 882ms
18:	learn: 0.3132016	total: 17ms	remaining: 8

In [85]:
score_df = pd.DataFrame.from_dict(scores).T

In [86]:
score_df

,MSE,RMSE,MAE,R2
LinearRegression,0.049582,0.222671,0.162846,0.843033
Ridge,0.049571,0.222646,0.162793,0.843068
Lasso,0.315984,0.562125,0.429485,-0.000344
DecisionTreeRegressor,0.049796,0.223150,0.144856,0.842356
SVR,0.039569,0.198919,0.143812,0.874733
GradientBoostingRegressor,0.030445,0.174485,0.127159,0.903617
RandomForestRegressor,0.028223,0.167996,0.112568,0.910653
AdaBoostRegressor,0.048659,0.220588,0.169438,0.845956
XGBRegressor,0.029004,0.170305,0.111988,0.908180
CatBoostRegressor,0.027169,0.164831,0.113981,0.913987


In [93]:
rt = automac.autotune.RegressionTuner(x=x_s , y=y_transformed)


In [95]:
tuner = rt.tune()

🚀 Starting Parallel Regression Tuning for: ['rf', 'xgb', 'lgb', 'cat', 'ada', 'dt', 'svm', 'knn']
✅ Tuning complete!


In [98]:
tuner

{'RandomForest': ({'n_estimators': 367,
   'max_depth': 26,
   'min_samples_split': 3},
  0.026411244355693544),
 'XGBoost': ({'n_estimators': 237,
   'learning_rate': 0.019641970874481877,
   'max_depth': 10,
   'subsample': 0.5951790412625925},
  0.02497368591429559),
 'LightGBM': ({'n_estimators': 594,
   'learning_rate': 0.013110298768911692,
   'num_leaves': 239},
  0.026702286688606077),
 'CatBoost': ({'iterations': 682,
   'depth': 8,
   'learning_rate': 0.12652258428307456},
  0.02525169532949082),
 'AdaBoost': ({'n_estimators': 142,
   'learning_rate': 0.5801159711340872,
   'loss': 'square'},
  0.04243367075029896),
 'DecisionTree': ({'max_depth': 6, 'criterion': 'squared_error'},
  0.041428303683500506),
 'SVM': ({'C': 1.9385488953385204,
   'epsilon': 0.09011252083486063,
   'kernel': 'rbf'},
  0.039524708599615155),
 'KNN': ({'n_neighbors': 6, 'weights': 'distance'}, 0.04472574051885287)}

In [105]:
tuner['KNN'][0]

{'n_neighbors': 6, 'weights': 'distance'}

In [114]:
from sklearn.neighbors import KNeighborsRegressor
models2 = [RandomForestRegressor(**tuner['RandomForest'][0]) , XGBRegressor(**tuner['XGBoost'][0]) , LGBMRegressor(**tuner['LightGBM'][0]) , CatBoostRegressor(**tuner['CatBoost'][0]) , AdaBoostRegressor(**tuner['AdaBoost'][0]) , DecisionTreeRegressor(**tuner['DecisionTree'][0]) , SVR(**tuner['SVM'][0]) , KNeighborsRegressor(**tuner['KNN'][0])]

names2 =list(tuner.keys())

In [136]:
scores2 = {}
traned_models = []
for i in range(len(models2)):
    score2 = ra.regression_metrics(model=models2[i])
    traned_models.append(models2[i]) 
    scores2[names2[i]] = score2

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000148 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 409
[LightGBM] [Info] Number of data points in the train set: 2487, number of used features: 12
[LightGBM] [Info] Start training from score 1.043280
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

In [123]:
score_df2 = pd.DataFrame.from_dict(scores2).T
score_df2

,MSE,RMSE,MAE,R2
RandomForest,0.027509,0.165858,0.110808,0.912912
XGBoost,0.026561,0.162975,0.109459,0.915914
LightGBM,0.028150,0.167780,0.113669,0.910883
CatBoost,0.027693,0.166413,0.113018,0.912329
AdaBoost,0.049407,0.222276,0.169142,0.843589
DecisionTree,0.040879,0.202185,0.147933,0.870585
SVM,0.039021,0.197536,0.142718,0.876469
KNN,0.050145,0.223930,0.149158,0.841252


In [129]:
score_df2.sort_values(by='MAE')

,MSE,RMSE,MAE,R2
XGBoost,0.026561,0.162975,0.109459,0.915914
RandomForest,0.027509,0.165858,0.110808,0.912912
CatBoost,0.027693,0.166413,0.113018,0.912329
LightGBM,0.028150,0.167780,0.113669,0.910883
SVM,0.039021,0.197536,0.142718,0.876469
DecisionTree,0.040879,0.202185,0.147933,0.870585
KNN,0.050145,0.223930,0.149158,0.841252
AdaBoost,0.049407,0.222276,0.169142,0.843589


In [131]:
for model in models2:
    ra.plot_regression_results(model)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000113 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 408
[LightGBM] [Info] Number of data points in the train set: 2487, number of used features: 12
[LightGBM] [Info] Start training from score 1.043498
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

0:	learn: 0.5159454	total: 3.2ms	remaining: 2.18s
1:	learn: 0.4683460	total: 4.93ms	remaining: 1.68s
2:	learn: 0.4262810	total: 6.46ms	remaining: 1.46s
3:	learn: 0.3893749	total: 7.9ms	remaining: 1.34s
4:	learn: 0.3577620	total: 9.32ms	remaining: 1.26s
5:	learn: 0.3311119	total: 11.2ms	remaining: 1.26s
6:	learn: 0.3088451	total: 12.9ms	remaining: 1.24s
7:	learn: 0.2888375	total: 14.5ms	remaining: 1.23s
8:	learn: 0.2720891	total: 16ms	remaining: 1.2s
9:	learn: 0.2580564	total: 17.8ms	remaining: 1.19s
10:	learn: 0.2453420	total: 19.5ms	remaining: 1.19s
11:	learn: 0.2352589	total: 21.3ms	remaining: 1.19s
12:	learn: 0.2271975	total: 22.9ms	remaining: 1.18s
13:	learn: 0.2195340	total: 24.9ms	remaining: 1.19s
14:	learn: 0.2127885	total: 26.8ms	remaining: 1.19s
15:	learn: 0.2070481	total: 28.4ms	remaining: 1.18s
16:	learn: 0.2022779	total: 30.7ms	remaining: 1.2s
17:	learn: 0.1974682	total: 32.9ms	remaining: 1.21s
18:	learn: 0.1937296	total: 35.1ms	remaining: 1.23s
19:	learn: 0.1900679	total: 

In [132]:
scaler

StandardScaler()

In [134]:
encoder

,cols,"['agePossession', 'luxury_category', ...]"
,verbose,0
,drop_invariant,False
,return_df,True
,handle_missing,'value'
,handle_unknown,'value'
,min_samples_leaf,20
,smoothing,10
,hierarchy,None
,min_group_size,None
,min_group_name,None


In [135]:
tuner

{'RandomForest': ({'n_estimators': 367,
   'max_depth': 26,
   'min_samples_split': 3},
  0.026411244355693544),
 'XGBoost': ({'n_estimators': 237,
   'learning_rate': 0.019641970874481877,
   'max_depth': 10,
   'subsample': 0.5951790412625925},
  0.02497368591429559),
 'LightGBM': ({'n_estimators': 594,
   'learning_rate': 0.013110298768911692,
   'num_leaves': 239},
  0.026702286688606077),
 'CatBoost': ({'iterations': 682,
   'depth': 8,
   'learning_rate': 0.12652258428307456},
  0.02525169532949082),
 'AdaBoost': ({'n_estimators': 142,
   'learning_rate': 0.5801159711340872,
   'loss': 'square'},
  0.04243367075029896),
 'DecisionTree': ({'max_depth': 6, 'criterion': 'squared_error'},
  0.041428303683500506),
 'SVM': ({'C': 1.9385488953385204,
   'epsilon': 0.09011252083486063,
   'kernel': 'rbf'},
  0.039524708599615155),
 'KNN': ({'n_neighbors': 6, 'weights': 'distance'}, 0.04472574051885287)}

In [138]:
traned_models[0]

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",367
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",26
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",3
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number

In [139]:
import pickle

with open('encoder.pkl', 'wb') as file:
    pickle.dump(encoder, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

with open('models.pkl', 'wb') as file:
    pickle.dump(traned_models, file)

In [140]:
df

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,0.0,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,0.0,High,Mid Floor
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3549,flat,sector 84,0.37,2.0,2.0,1,Relatively New,532.0,0.0,0.0,0.0,Medium,Mid Floor
3550,house,sector 109,6.00,5.0,5.0,3+,Relatively New,6228.0,1.0,1.0,0.0,High,Low Floor
3551,flat,sector 2,0.60,1.0,1.0,1,Moderately Old,665.0,0.0,0.0,1.0,Medium,Mid Floor
3552,house,sector 43,15.50,5.0,6.0,3,Moderately Old,5490.0,1.0,1.0,0.0,Medium,Mid Floor


In [141]:
with open('df.pkl', 'wb') as file:
    pickle.dump(df, file)

In [142]:
encoder

,cols,"['agePossession', 'luxury_category', ...]"
,verbose,0
,drop_invariant,False
,return_df,True
,handle_missing,'value'
,handle_unknown,'value'
,min_samples_leaf,20
,smoothing,10
,hierarchy,None
,min_group_size,None
,min_group_name,None


In [145]:
traned_models[:4]

[RandomForestRegressor(max_depth=26, min_samples_split=3, n_estimators=367),
 XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.019641970874481877, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=237, n_jobs=None,
              num_parallel_tree=None, ...),
 LGBMRegressor(learning_rate=0.013110298768911692, n_estimators=594,
               num_leaves=239),
 CatBoostRegressor(depth=8, iterat

In [152]:
encoder.feature_names_out_


array(['agePossession', 'luxury_category', 'floor_category', 'sector',
       'property_type', 'balcony'], dtype=object)

In [166]:
row = X.iloc[[1]]  # double brackets preserve DataFrame structure
row_encoded = encoder.transform(row)
row_e = scaler.transform(row['built_up_area'].values.reshape(-1 ,1))

In [160]:
row

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
1,flat,sector 89,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Mid Floor


In [168]:
row_e

array([[1226.]])

In [172]:
scaler.transform(row)

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- agePossession
- balcony
- bathroom
- bedRoom
- floor_category
- ...


In [176]:
x_s.columns

Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category'],
      dtype='str')

In [ ]:
Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category'],
      dtype='str')